In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import logging
import multiprocessing
from multiprocessing import Pool
import matplotlib.pyplot as plt
from scipy.stats import norm
from userlist import *
import re
from AbundProcessFunctions import *

# This code is used to plot abundance patterns for different elements.
- [X/H] sorted by atomic properties

### First, read the raw results from `output_abundances_result_path`

- df: DataFrame loaded from `output_abundances_result_path` and post-processed
    - Contains line-by-line records for all ionization states
    - Newly added columns:
        - base_element : e.g. "Fe 1" → "Fe"
        - ion          : e.g. "Fe 1" → 1 (defaults to 1 if missing)
        - Z            : atomic number (mapped via ATOMIC_Z, missing values set to 9999)
    - Ionization states are NOT merged (each ion remains a separate row)
    - No condensation temperature (Tc) processing is applied

- df_combined: DataFrame obtained by grouping and merging `df` by `base_element` (element-level)
    - Merging rules:
        - [X/H]       → mean over ionization states
        - std_[X/H]   → mean over ionization states (can be changed to max or standard error if needed)
        - n_lines     → sum over ionization states
    - Newly added columns:
        - Z            : atomic number (mapped via ATOMIC_Z)
    - Ionization-state dimension is removed (one row per element)
    - No condensation temperature (Tc) processing is applied

- order (internal variable): x-axis ordering derived from `df`
    - Computed by sorting on (Z, ion, base_element, element)
    - Used to set df["element"] as an ordered Categorical to ensure a stable x-axis in plots

- Plotting functions:
    - plot_with_ions(df)
        → Plot abundances distinguishing ionization states
        → x-axis: element = "X ion"
        → order: (Z, ion)

    - plot_combined(df_combined)
        → Plot abundances with ionization states merged
        → x-axis: base_element
        → order: Z

In [ ]:
# ---------- Load & preprocess ----------
df = pd.read_csv(output_abundances_result_path_linebyline_q2)
os.makedirs(abundance_plot_folder, exist_ok=True)

In [ ]:
# ---------- Plotting functions ----------

# ========== fig01 ==========
def plot_xh_by_Z_with_ion(df_in, target, outdir):
    """
    fig01: [X/H] ordered by (Z, ion), separating Fe from other elements.
    The x-axis keeps ionization-stage labels; n_lines is annotated for each ion.
    Required columns in df_in:
        element, base_element, ion, Z, [X/H], std_[X/H], n_lines
    """
    mkdir(outdir)
    d = df_in.copy()
    order = d.sort_values(["Z","ion","base_element","element"])["element"].tolist()
    d["element"] = pd.Categorical(d["element"], categories=order, ordered=True)
    d = d.sort_values("element")

    plt.figure(figsize=(12,6), dpi=192)
    mask_fe = d["element"].str.startswith("Fe")
    plt.errorbar(d.loc[~mask_fe,"element"], d.loc[~mask_fe,"[X/H]"],
                 yerr=d.loc[~mask_fe,"std_[X/H]"], fmt="o",
                 color="blue", ecolor="gray", elinewidth=1.5, capsize=4, label="Other elements")
    plt.errorbar(d.loc[ mask_fe,"element"], d.loc[ mask_fe,"[X/H]"],
                 yerr=d.loc[ mask_fe,"std_[X/H]"], fmt="o",
                 color="red",  ecolor="gray", elinewidth=1.5, capsize=4, label="Fe")

    for _, r in d.iterrows():
        plt.text(r["element"], r["[X/H]"]+0.03, f"{int(r['n_lines'])}",
                 ha="center", va="bottom", fontsize=9)

    plt.axhline(0, color="k", ls="--", lw=1)
    plt.xlabel("Element (with ionization stage)", fontsize=14)
    plt.ylabel("[X/H] (dex)", fontsize=14)
    plt.title(f"{target}, [X/H] sorted by Z, then ion", fontsize=16)
    plt.xticks(rotation=45, ha="right")
    plt.grid(alpha=0.3)
    plt.legend(fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, "01_XH_ordered_by_Z_with_ion.png"), dpi=192)

# ========== fig02 ==========
def plot_xh_by_Z_combined(df_in, target, outdir):
    """
    fig02: [X/H] with ionization states combined (error = SEM across ions,
    or the intrinsic uncertainty for a single ion), ordered by Z.
    Required columns in df_in:
        base_element, [X/H], std_[X/H], n_lines, Z
    """
    mkdir(outdir)
    d = df_in.sort_values("Z").copy()
    mask_fe = d["base_element"]=="Fe"

    plt.figure(figsize=(12,6), dpi=192)
    plt.errorbar(d.loc[~mask_fe,"base_element"], d.loc[~mask_fe,"[X/H]"],
                 yerr=d.loc[~mask_fe,"std_[X/H]"], fmt="o", color="blue",
                 ecolor="gray", elinewidth=1.5, capsize=4, markersize=10, label="Other elements")
    plt.errorbar(d.loc[ mask_fe,"base_element"], d.loc[ mask_fe,"[X/H]"],
                 yerr=d.loc[ mask_fe,"std_[X/H]"], fmt="o", color="red",
                 ecolor="gray", elinewidth=1.5, capsize=4, markersize=10, label="Fe")

    for _, r in d.iterrows():
        plt.text(r["base_element"], r["[X/H]"]+0.03, f"{int(r['n_lines'])}",
                 ha="center", va="bottom", fontsize=12)

    plt.axhline(0, color="k", ls="--", lw=1)
    plt.ylabel("[X/H] (dex)", fontsize=14)
    plt.title(f"{target}, [X/H] sorted by Z (ions combined)", fontsize=16)
    plt.xticks(d["base_element"].unique(), rotation=45, ha="right", fontsize=13)
    plt.yticks(fontsize=12)
    plt.grid(alpha=0.3)
    plt.legend(fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, "02_XH_ordered_by_Z.png"), dpi=192)

# ========== fig03 ==========
def plot_xh_by_Tc_ordered(target_df, target, outdir):
    """
    fig03: [X/H] ordered by condensation temperature Tc
    (x-axis shows element names; no reference comparison).
    Required columns in target_df:
        Element, [X/H], std_[X/H], n_lines, Tc_K
    """
    mkdir(outdir)
    d = target_df.dropna(subset=["Tc_K"]).copy().sort_values(["Tc_K","Element"])
    xticks = d["Element"].tolist()

    plt.figure(figsize=(12,6), dpi=192)
    plt.errorbar(xticks, d["[X/H]"], yerr=d["std_[X/H]"],
                 fmt="o", color="blue", ecolor="gray", elinewidth=1.5, capsize=4)
    for _, r in d.iterrows():
        plt.text(r["Element"], r["[X/H]"]+0.03, f"{int(r.get('n_lines',0))}",
                 ha="center", va="bottom", fontsize=9)
    plt.axhline(0, color="k", ls="--", lw=1)
    plt.ylabel("[X/H] (dex)", fontsize=14)
    plt.title(f"{target}, [X/H] ordered by T$_c$", fontsize=16)
    plt.xticks(xticks, rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, "03_XH_ordered_by_Tc.png"), dpi=192)

# ========== fig04 ==========
def plot_xh_vs_Tc_labels(target_df, target, outdir):
    """
    fig04: [X/H] vs Tc (numeric x-axis), with element labels above each point;
    no reference comparison.
    Required columns in target_df:
        Element, [X/H], std_[X/H], Tc_K
    """
    mkdir(outdir)
    d = target_df.dropna(subset=["Tc_K"]).copy().sort_values(["Tc_K","Element"])
    plt.figure(figsize=(12,6), dpi=192)
    plt.errorbar(d["Tc_K"], d["[X/H]"], yerr=d["std_[X/H]"],
                 fmt="o", color="blue", ecolor="gray", elinewidth=1.5, capsize=4)
    for _, r in d.iterrows():
        plt.text(r["Tc_K"], r["[X/H]"]+0.03, r["Element"], ha="center", va="bottom", fontsize=10)
    plt.axhline(0, color="k", ls="--", lw=1)
    plt.xlabel("Condensation temperature T$_c$ (K)", fontsize=14)
    plt.ylabel("[X/H] (dex)", fontsize=14)
    plt.title(f"{target}, [X/H] vs T$_c$", fontsize=16)
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, "04_XH_vs_Tc_labels_Element.png"), dpi=192)

# ========== fig05 ==========
def plot_xh_by_Tc_with_ref(target_df, ref_df, target, ref_name, outdir):
    """
    fig05: [X/H] ordered by Tc (x-axis = element names), with a reference comparison.
    Required columns in both tables:
        Element, [X/H], std_[X/H], n_lines, Tc_K
    """
    mkdir(outdir)
    td = target_df.dropna(subset=["Tc_K"]).copy().sort_values(["Tc_K","Element"])
    rd = ref_df.dropna(subset=["Tc_K"]).copy().sort_values(["Tc_K","Element"])
    rd = rd[rd["Element"].isin(td["Element"])]
    td = td[td["Element"].isin(rd["Element"])]
    xticks = td["Element"].tolist()

    plt.figure(figsize=(12,6), dpi=192)
    plt.errorbar(xticks, td["[X/H]"], yerr=td["std_[X/H]"],
                 fmt="o", color="blue", ecolor="gray", elinewidth=1.5, capsize=4, label="Target")
    plt.errorbar(xticks, rd["[X/H]"], yerr=rd["std_[X/H]"],
                 fmt="o", mfc="white", mec="green", ecolor="green",
                 elinewidth=1.2, capsize=3, label=f"Ref ({ref_name})")
    for _, r in td.iterrows():
        plt.text(r["Element"], r["[X/H]"]+0.03, f"{int(r.get('n_lines',0))}",
                 ha="center", va="bottom", fontsize=9)
    plt.axhline(0, color="k", ls="--", lw=1)
    plt.ylabel("[X/H] (dex)", fontsize=14)
    plt.title(f"{target}, [X/H] ordered by T$_c$ with Ref", fontsize=16)
    plt.xticks(xticks, rotation=45, ha="right")
    plt.legend(fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, "05_XH_ordered_by_Tc_withRef.png"), dpi=192)

# ========== fig06 ==========
def plot_xfe_by_Tc_with_ref(target_xfe_df, ref_xfe_df, target, ref_name, outdir):
    """
    fig06: [X/Fe] ordered by Tc (x-axis = element names), with a reference comparison.
    Required columns in both tables:
        Element, [X/Fe], e_[X/Fe], Tc_K
    """
    mkdir(outdir)
    td = target_xfe_df.dropna(subset=["Tc_K"]).copy().sort_values(["Tc_K","Element"])
    rd = ref_xfe_df.dropna(subset=["Tc_K"]).copy().sort_values(["Tc_K","Element"])
    td = td[td["Element"].isin(rd["Element"])]
    rd = rd[rd["Element"].isin(td["Element"])]
    xticks = td["Element"].tolist()

    plt.figure(figsize=(12,6), dpi=192)
    plt.errorbar(xticks, td["[X/Fe]"], yerr=td["e_[X/Fe]"],
                 fmt="o", color="blue", ecolor="gray", elinewidth=1.5, capsize=4, label="Target")
    plt.errorbar(xticks, rd["[X/Fe]"], yerr=rd["e_[X/Fe]"],
                 fmt="o", mfc="white", mec="green", ecolor="green",
                 elinewidth=1.2, capsize=3, label=f"Ref ({ref_name})")
    plt.axhline(0, color="k", ls="--", lw=1)
    plt.ylabel("[X/Fe] (dex)", fontsize=14)
    plt.title(f"{target}, [X/Fe] ordered by T$_c$ with Ref", fontsize=16)
    plt.xticks(xticks, rotation=45, ha="right")
    plt.legend(fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, "06_XFe_ordered_by_Tc_withRef.png"), dpi=192)

# ========== fig07 ==========
def plot_xh_vs_Tc_with_ref_labels(target_df, ref_df, target, ref_name, outdir):
    """
    fig07: [X/H] vs Tc with element labels above points and a reference comparison.
    Required columns in both tables:
        Element, [X/H], std_[X/H], Tc_K
    """
    mkdir(outdir)
    td = target_df.dropna(subset=["Tc_K"]).copy().sort_values(["Tc_K","Element"])
    rd = ref_df.dropna(subset=["Tc_K"]).copy().sort_values(["Tc_K","Element"])
    td, rd = common_elements_on_Tc(td, rd)

    plt.figure(figsize=(12,6), dpi=192)
    plt.errorbar(td["Tc_K"], td["[X/H]"], yerr=td["std_[X/H]"],
                 fmt="o", color="blue", ecolor="gray", elinewidth=1.5, capsize=4, label="Target")
    plt.errorbar(rd["Tc_K"], rd["[X/H]"], yerr=rd["std_[X/H]"],
                 fmt="o", mfc="white", mec="green", ecolor="green",
                 elinewidth=1.2, capsize=3, label=f"Ref ({ref_name})")

    for _, r in td.iterrows():
        plt.text(r["Tc_K"], r["[X/H]"]+0.03, r["Element"], ha="center", va="bottom", fontsize=10)
    for _, r in rd.iterrows():
        plt.text(r["Tc_K"], r["[X/H]"]-0.03, r["Element"], ha="center", va="top", fontsize=10, color="green")

    plt.axhline(0, color="k", ls="--", lw=1)
    plt.xlabel("Condensation temperature T$_c$ (K)", fontsize=14)
    plt.ylabel("[X/H] (dex)", fontsize=14)
    plt.title(f"{target}, [X/H] vs T$_c$ with Ref", fontsize=16)
    plt.legend(fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, "07_XH_vs_Tc_labels_withRef.png"), dpi=192)

# ========== fig08 ==========
def plot_xfe_vs_Tc_with_ref_labels(target_xfe_df, ref_xfe_df, target, ref_name, outdir):
    """
    fig08: [X/Fe] vs Tc with element labels above points and a reference comparison.
    Required columns in both tables:
        Element, [X/Fe], e_[X/Fe], Tc_K
    """
    mkdir(outdir)
    td = target_xfe_df.dropna(subset=["Tc_K"]).copy().sort_values(["Tc_K","Element"])
    rd = ref_xfe_df.dropna(subset=["Tc_K"]).copy().sort_values(["Tc_K","Element"])
    td, rd = common_elements_on_Tc(td, rd)

    plt.figure(figsize=(12,6), dpi=192)
    plt.errorbar(td["Tc_K"], td["[X/Fe]"], yerr=td["e_[X/Fe]"],
                 fmt="o", color="blue", ecolor="gray", elinewidth=1.5, capsize=4, label="Target")
    plt.errorbar(rd["Tc_K"], rd["[X/Fe]"], yerr=rd["e_[X/Fe]"],
                 fmt="o", mfc="white", mec="green", ecolor="green",
                 elinewidth=1.2, capsize=3, label=f"Ref ({ref_name})")

    for _, r in td.iterrows():
        plt.text(r["Tc_K"], r["[X/Fe]"]+0.03, r["Element"], ha="center", va="bottom", fontsize=10)
    for _, r in rd.iterrows():
        plt.text(r["Tc_K"], r["[X/Fe]"]-0.03, r["Element"], ha="center", va="top", fontsize=10, color="green")

    plt.axhline(0, color="k", ls="--", lw=1)
    plt.xlabel("Condensation temperature T$_c$ (K)", fontsize=14)
    plt.ylabel("[X/Fe] (dex)", fontsize=14)
    plt.title(f"{target}, [X/Fe] vs T$_c$ with Ref", fontsize=16)
    plt.legend(fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, "08_XFe_vs_Tc_labels_withRef.png"), dpi=192)

In [ ]:
# Parse element token -> base_element, ion, Z
tmp = df["element"].apply(parse_element_token)
df["base_element"] = tmp.apply(lambda x: x[0])
df["ion"] = tmp.apply(lambda x: x[1])
df["Z"] = df["base_element"].map(ATOMIC_Z).fillna(9999)
#print(df)

# Combine ionization states
df_combined = (
    df.groupby("base_element", as_index=False)
      .apply(combine_ions_sem)
)

# Add atomic number
df_combined["Z"] = df_combined["base_element"].map(ATOMIC_Z)
#print(df_combined)

# fig01 / fig02
plot_xh_by_Z_with_ion(df, target=target, outdir=abundance_plot_folder)
plot_xh_by_Z_combined(df_combined, target=target, outdir=abundance_plot_folder)

## Sorted by Condensation temperature

### On Uncertainties

- **Reference (Meléndez et al., A46_table1/2)**
  - **Inputs:**
    - `A46_table1` provides \([{\rm Fe/H}]\) and \(e_{[{\rm Fe/H}]}\).
    - `A46_table2` provides \([{\rm X/Fe}]\) and \(e_{[{\rm X/Fe}]}\).
  - **Conversion to \([{\rm X/H}]\):**
    \[
      [{\rm X/H}] = [{\rm X/Fe}] + [{\rm Fe/H}]
    \]
  - **Uncertainty propagation** (assuming independent errors, combined in quadrature):
    \[
      \sigma_{[{\rm X/H}]} = \sqrt{\sigma_{[{\rm X/Fe}]}^2 + \sigma_{[{\rm Fe/H}]}^2}
    \]

- **Ionization-stage handling**
  - Ionization stages are first combined by taking the mean abundance, with the uncertainty given by the **SEM across ionization stages**.
  - If \([{\rm X/Fe}]\) is required, the difference is then computed and uncertainties are combined **in quadrature**.

In [ ]:
# ========== Prepare basic data for plotting ==========
tc_df = load_tc(TC_PATH)

# Target star data with ions combined
# df_combined columns: Element, [X/H], std_[X/H], n_lines, Z
target_xh = ensure_element_col(df_combined)[["Element","[X/H]","std_[X/H]","n_lines","Z"]].copy()
target_xh = add_tc_to(target_xh, tc_df)

# Derive [X/Fe] for the target (ions combined)
fe_row = target_xh.loc[target_xh["Element"]=="Fe"].iloc[0] if (target_xh["Element"]=="Fe").any() else None
if fe_row is not None:
    target_xfe = target_xh.copy()
    target_xfe["[X/Fe]"] = target_xfe["[X/H]"] - fe_row["[X/H]"]
    # Uncertainty for combined ions:
    # keep std_[X/H] as the y-error; optionally propagate with Fe uncertainty
    target_xfe["e_[X/Fe]"] = np.sqrt(
        np.where(np.isfinite(target_xfe["std_[X/H]"]), target_xfe["std_[X/H]"]**2, 0.0) +
        (fe_row["std_[X/H]"]**2 if np.isfinite(fe_row["std_[X/H]"]) else 0.0)
    )
else:
    # Fallback if Fe is missing (should be rare)
    target_xfe = target_xh.assign(**{"[X/Fe]": np.nan, "e_[X/Fe]": np.nan})

# ========== Build reference abundances (generic paths) ==========
if REF_NAME == "Bedell2018":
    bedell_table2 = os.path.join(ispec_dir, "input", "abundances", "Bedell2018", "Bedell2018_table2.csv")
    bedell_feh_table = os.path.join(ispec_dir, "input", "abundances", "Bedell2018", "abundance_with_gaia.csv")

    ref_xh, ref_xfe = build_ref_for_star_choice(
        star=REF_STAR,
        ref_name="Bedell2018",
        bedell_table2_path=bedell_table2,
        feh_table_path=bedell_feh_table,
    )

elif REF_NAME == "Melendez2025":
    a1_path = os.path.join(ispec_dir, "input", "abundances", "Melendez2025", "A46_tablea1_raw.csv")
    a2_path = os.path.join(ispec_dir, "input", "abundances", "Melendez2025", "A46_table2_raw.csv")

    ref_xh, ref_xfe = build_ref_for_star_choice(
        star=REF_STAR,
        ref_name="Melendez2025",
        a46_a1_path=a1_path,
        a46_a2_path=a2_path,
    )

# Add Tc to reference tables
ref_xh  = add_tc_to(ref_xh,  tc_df)
ref_xfe = add_tc_to(ref_xfe, tc_df)

# ========== Plotting ==========
# fig03–fig08
plot_xh_by_Tc_ordered(target_xh, target=target, outdir=abundance_plot_folder)
plot_xh_vs_Tc_labels(target_xh, target=target, outdir=abundance_plot_folder)
plot_xh_by_Tc_with_ref(target_xh, ref_xh, target=target, ref_name=REF_STAR, outdir=abundance_plot_folder)
plot_xfe_by_Tc_with_ref(target_xfe, ref_xfe, target=target, ref_name=REF_STAR, outdir=abundance_plot_folder)
plot_xh_vs_Tc_with_ref_labels(target_xh, ref_xh, target=target, ref_name=REF_STAR, outdir=abundance_plot_folder)
plot_xfe_vs_Tc_with_ref_labels(target_xfe, ref_xfe, target=target, ref_name=REF_STAR, outdir=abundance_plot_folder)